# Zadanie indywidualne: prosta sieć neuronowa dla zbioru Iris w TensorFlow / Keras

**Temat:** klasyfikacja wieloklasowa na zbiorze Iris  
**Technologie:** Python, TensorFlow / Keras, NumPy, pandas, scikit-learn, matplotlib  
**Cel:** zbudować, wytrenować i zinterpretować prostą sieć neuronową rozpoznającą gatunki irysów na podstawie 4 cech liczbowych.

Notebook zawiera pełne rozwiązanie:

1. wczytanie danych,
2. przygotowanie danych,
3. budowę modelu,
4. kompilację i trening,
5. ocenę,
6. predykcję,
7. analizę błędów,
8. wykresy uczenia,
9. eksperyment z architekturą,
10. odpowiedzi na pytania kontrolne.

## 0. Przygotowanie środowiska

W środowisku Google Colab TensorFlow jest zwykle dostępny od razu. Lokalnie można go doinstalować poleceniem:

```bash
pip install tensorflow scikit-learn pandas numpy matplotlib
```

In [ ]:
import os
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Dla powtarzalności wyników.
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)

## 1. Wczytanie danych

Zbiór Iris zawiera 150 próbek, 4 cechy wejściowe i 3 klasy: `setosa`, `versicolor`, `virginica`.

In [ ]:
iris = load_iris()

X = iris.data
y = iris.target

feature_names = iris.feature_names
class_names = iris.target_names

df = pd.DataFrame(X, columns=feature_names)
df["target"] = y
df["class_name"] = [class_names[i] for i in y]

print("Opis zbioru:")
print(iris.DESCR[:900], "...\n")

print("Liczba próbek:", X.shape[0])
print("Liczba cech wejściowych:", X.shape[1])
print("Dostępne klasy:", list(class_names))
print("Kształt X:", X.shape)
print("Kształt y:", y.shape)

df.head()

In [ ]:
# Rozkład klas w zbiorze.
class_distribution = df["class_name"].value_counts().sort_index()
class_distribution

## 2. Przygotowanie danych

Dane dzielimy na część treningową i testową w proporcji 80/20.  
Używamy `stratify=y`, żeby zachować podobny rozkład klas w obu zbiorach.

Bardzo ważna zasada: `StandardScaler` dopasowujemy tylko na danych treningowych. Zbiór testowy wyłącznie transformujemy. Dzięki temu nie podglądamy danych testowych podczas treningu.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

print("\nŚrednie cech po skalowaniu na zbiorze treningowym:")
print(np.round(X_train_scaled.mean(axis=0), 4))

print("\nOdchylenia standardowe cech po skalowaniu na zbiorze treningowym:")
print(np.round(X_train_scaled.std(axis=0), 4))

## 3. Budowa modelu bazowego

Model bazowy ma strukturę:

- wejście: 4 cechy,
- warstwa ukryta `Dense(16, activation="relu")`,
- warstwa ukryta `Dense(8, activation="relu")`,
- wyjście: `Dense(3, activation="softmax")`.

Ostatnia warstwa ma 3 neurony, ponieważ mamy 3 klasy irysów.

In [ ]:
def build_model(hidden_layers=(16, 8), learning_rate=0.001, seed=SEED):
    """Buduje prostą sieć neuronową dla klasyfikacji zbioru Iris.

    Parameters
    ----------
    hidden_layers : tuple[int, ...]
        Liczba neuronów w kolejnych warstwach ukrytych.
    learning_rate : float
        Współczynnik uczenia optymalizatora Adam.
    seed : int
        Ziarno losowości dla inicjalizacji wag.

    Returns
    -------
    keras.Model
        Skompilowany model TensorFlow / Keras.
    """
    tf.random.set_seed(seed)

    model = keras.Sequential(name="iris_neural_network")
    model.add(layers.Input(shape=(4,), name="input_4_features"))

    for index, units in enumerate(hidden_layers, start=1):
        model.add(layers.Dense(units, activation="relu", name=f"dense_relu_{index}"))

    model.add(layers.Dense(3, activation="softmax", name="output_softmax_3_classes"))

    # sparse_categorical_crossentropy stosujemy wtedy, gdy etykiety klas
    # są zapisane jako liczby całkowite, np. 0, 1, 2.
    # Gdyby etykiety były zakodowane one-hot, użylibyśmy categorical_crossentropy.
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

base_model = build_model(hidden_layers=(16, 8))
base_model.summary()

## 4. Trenowanie modelu

Model trenujemy przez 50 epok. Część zbioru treningowego, 20%, zostaje użyta jako zbiór walidacyjny.

In [ ]:
EPOCHS = 50
BATCH_SIZE = 8

history = base_model.fit(
    X_train_scaled,
    y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    verbose=1
)

## 5. Ocena modelu na danych testowych

Sprawdzamy model na danych, których nie widział podczas uczenia.

In [ ]:
test_loss, test_accuracy = base_model.evaluate(X_test_scaled, y_test, verbose=0)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

### Krótka interpretacja oceny

- **Czy model osiągnął wysoką dokładność?**  
  Dla zbioru Iris prosta sieć zwykle osiąga wysoką dokładność, często powyżej 90%, choć dokładny wynik zależy od inicjalizacji wag i podziału danych.

- **Czy wynik jest lepszy niż losowe zgadywanie?**  
  Tak. Losowe zgadywanie dla 3 klas daje średnio około 33% trafień. Wynik znacząco wyższy od 0.33 oznacza, że model znalazł strukturę w danych.

- **Czy funkcja straty jest niska?**  
  Im bliżej zera, tym lepiej. W praktyce patrzymy nie tylko na wartość `loss`, ale też na stabilność krzywych uczenia i accuracy.

- **Czy model prawdopodobnie nauczył się sensownej zależności?**  
  Tak, jeżeli dokładność testowa jest wysoka, a macierz pomyłek pokazuje niewiele błędów. Model nie „rozumie” kwiatów, lecz rozpoznaje zależności liczbowe między cechami a klasami.

## 6. Predykcja

Model zwraca prawdopodobieństwa przynależności próbki do każdej klasy. Funkcja `argmax` wybiera klasę o najwyższym prawdopodobieństwie.

In [ ]:
y_proba = base_model.predict(X_test_scaled)
y_pred = np.argmax(y_proba, axis=1)

predictions_df = pd.DataFrame({
    "true_label": y_test,
    "true_class": [class_names[i] for i in y_test],
    "predicted_label": y_pred,
    "predicted_class": [class_names[i] for i in y_pred],
    "max_probability": np.max(y_proba, axis=1)
})

print("Pierwsze 10 wyników: prawdziwa klasa | przewidziana klasa")
display(predictions_df.head(10))

In [ ]:
# Prawdopodobieństwa klas dla pierwszych 10 próbek testowych.
proba_df = pd.DataFrame(y_proba, columns=[f"P({name})" for name in class_names])
proba_df.insert(0, "true_class", [class_names[i] for i in y_test])
proba_df.insert(1, "predicted_class", [class_names[i] for i in y_pred])
proba_df.head(10)

## 7. Analiza wyników predykcji

W tej części tworzymy macierz pomyłek i raport klasyfikacyjny.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=[f"true_{name}" for name in class_names],
    columns=[f"pred_{name}" for name in class_names]
)

cm_df

In [ ]:
plt.figure(figsize=(6, 5))
plt.imshow(cm)
plt.title("Macierz pomyłek - model bazowy")
plt.xlabel("Klasa przewidziana")
plt.ylabel("Klasa prawdziwa")
plt.xticks(ticks=np.arange(len(class_names)), labels=class_names, rotation=45)
plt.yticks(ticks=np.arange(len(class_names)), labels=class_names)
plt.colorbar()

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
plt.show()

In [ ]:
report_dict = classification_report(
    y_test,
    y_pred,
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report_dict).transpose()
report_df

In [ ]:
# Które klasy model rozpoznaje najlepiej?
recall_per_class = np.diag(cm) / cm.sum(axis=1)
recall_df = pd.DataFrame({
    "class": class_names,
    "recall": recall_per_class
}).sort_values("recall", ascending=False)

print("Rozpoznawanie klas według recall:")
display(recall_df)

# Które klasy najczęściej się mylą?
confusions = []
for true_idx, true_name in enumerate(class_names):
    for pred_idx, pred_name in enumerate(class_names):
        if true_idx != pred_idx and cm[true_idx, pred_idx] > 0:
            confusions.append({
                "true_class": true_name,
                "predicted_class": pred_name,
                "count": cm[true_idx, pred_idx]
            })

confusions_df = pd.DataFrame(confusions).sort_values("count", ascending=False) if confusions else pd.DataFrame(
    columns=["true_class", "predicted_class", "count"]
)

print("Najczęstsze pomyłki:")
display(confusions_df)

### Interpretacja macierzy pomyłek

W zbiorze Iris klasa **setosa** jest zwykle najłatwiejsza do rozpoznania, ponieważ jej cechy płatków są mocno odseparowane od pozostałych gatunków. Najczęstsze pomyłki, jeżeli wystąpią, pojawiają się zwykle między klasami **versicolor** i **virginica**, które są do siebie bardziej podobne w przestrzeni cech.

## 8. Wizualizacja procesu uczenia

Rysujemy funkcję straty i dokładność dla zbioru treningowego oraz walidacyjnego.

In [ ]:
def plot_training_history(history, title_suffix=""):
    """Rysuje wykres loss i accuracy dla historii treningu modelu."""
    history_df = pd.DataFrame(history.history)

    plt.figure(figsize=(8, 5))
    plt.plot(history_df["loss"], label="loss")
    plt.plot(history_df["val_loss"], label="val_loss")
    plt.title(f"Funkcja straty {title_suffix}")
    plt.xlabel("Epoka")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(history_df["accuracy"], label="accuracy")
    plt.plot(history_df["val_accuracy"], label="val_accuracy")
    plt.title(f"Dokładność {title_suffix}")
    plt.xlabel("Epoka")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

plot_training_history(history, title_suffix="- model bazowy")

### Interpretacja wykresów uczenia

- **Czy model konwerguje?**  
  Tak, jeżeli `loss` i `val_loss` maleją lub stabilizują się, a `accuracy` i `val_accuracy` rosną lub osiągają plateau.

- **Czy widać oznaki przeuczenia?**  
  Przeuczenie byłoby widoczne wtedy, gdy `loss` na treningu dalej maleje, a `val_loss` zaczyna rosnąć. Jeżeli krzywe treningowe i walidacyjne są blisko siebie, przeuczenie jest niewielkie.

- **Czy warto zwiększyć lub zmniejszyć liczbę epok?**  
  Jeżeli po pewnej liczbie epok metryki się stabilizują, dalszy trening może nie dawać istotnych korzyści. Dla Iris 50 epok zwykle wystarcza.

- **Czy model uczy się stabilnie?**  
  Stabilne uczenie oznacza brak gwałtownych skoków `loss` i stopniową poprawę accuracy. Przy małym zbiorze danych drobne wahania walidacji są normalne.

## 9. Eksperyment z architekturą

Porównujemy trzy modele:

| Model | Architektura |
|---|---|
| Bazowy | Dense(16, relu) → Dense(8, relu) → Dense(3, softmax) |
| Wariant A | Dense(8, relu) → Dense(3, softmax) |
| Wariant B | Dense(32, relu) → Dense(16, relu) → Dense(3, softmax) |

In [ ]:
def train_evaluate_model(model_name, hidden_layers, epochs=50, batch_size=8):
    """Trenuje i ocenia wariant modelu."""
    model = build_model(hidden_layers=hidden_layers)

    start_time = time.time()
    hist = model.fit(
        X_train_scaled,
        y_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_split=0.2,
        verbose=0
    )
    training_time = time.time() - start_time

    loss, accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)

    final_train_loss = hist.history["loss"][-1]
    final_val_loss = hist.history["val_loss"][-1]
    final_train_accuracy = hist.history["accuracy"][-1]
    final_val_accuracy = hist.history["val_accuracy"][-1]

    return {
        "model_name": model_name,
        "hidden_layers": hidden_layers,
        "test_loss": loss,
        "test_accuracy": accuracy,
        "final_train_loss": final_train_loss,
        "final_val_loss": final_val_loss,
        "final_train_accuracy": final_train_accuracy,
        "final_val_accuracy": final_val_accuracy,
        "training_time_sec": training_time,
        "model": model,
        "history": hist
    }

experiment_results = [
    train_evaluate_model("Bazowy", (16, 8), epochs=EPOCHS, batch_size=BATCH_SIZE),
    train_evaluate_model("Wariant A", (8,), epochs=EPOCHS, batch_size=BATCH_SIZE),
    train_evaluate_model("Wariant B", (32, 16), epochs=EPOCHS, batch_size=BATCH_SIZE)
]

comparison_df = pd.DataFrame([
    {
        "model_name": result["model_name"],
        "hidden_layers": str(result["hidden_layers"]),
        "test_loss": result["test_loss"],
        "test_accuracy": result["test_accuracy"],
        "final_train_loss": result["final_train_loss"],
        "final_val_loss": result["final_val_loss"],
        "final_train_accuracy": result["final_train_accuracy"],
        "final_val_accuracy": result["final_val_accuracy"],
        "training_time_sec": result["training_time_sec"]
    }
    for result in experiment_results
]).sort_values("test_accuracy", ascending=False)

comparison_df

In [ ]:
best_result = max(experiment_results, key=lambda item: item["test_accuracy"])

print("Najlepszy model według test_accuracy:")
print("Nazwa:", best_result["model_name"])
print("Architektura:", best_result["hidden_layers"])
print(f"Test accuracy: {best_result['test_accuracy']:.4f}")
print(f"Test loss: {best_result['test_loss']:.4f}")
print(f"Czas uczenia: {best_result['training_time_sec']:.4f} s")

In [ ]:
# Wykres porównawczy accuracy modeli.
plot_df = comparison_df.sort_values("model_name")

plt.figure(figsize=(8, 5))
plt.bar(plot_df["model_name"], plot_df["test_accuracy"])
plt.title("Porównanie dokładności testowej modeli")
plt.xlabel("Model")
plt.ylabel("Test accuracy")
plt.ylim(0, 1.05)
plt.grid(axis="y")
plt.tight_layout()
plt.show()

### Wnioski z eksperymentu

Najlepszy model należy wybrać na podstawie `test_accuracy`, `test_loss`, stabilności krzywych uczenia i czasu treningu.

W praktyce na małym zbiorze Iris bardzo duża sieć nie jest konieczna. Często model bazowy lub wariant B osiągają podobną dokładność, ale wariant B może trenować się trochę dłużej. Wariant A jest najprostszy i może być wystarczający, jeśli jego accuracy jest zbliżone do większych modeli.

Dobra decyzja modelowa nie polega wyłącznie na maksymalizacji accuracy. Liczy się też prostota, stabilność i brak nadmiernego przeuczenia.

## 10. Dodatkowe wyzwanie: własny przykład kwiatu

Sprawdzamy predykcję dla próbki:

```python
sample = [[5.1, 3.5, 1.4, 0.2]]
```

In [ ]:
sample = np.array([[5.1, 3.5, 1.4, 0.2]])
sample_scaled = scaler.transform(sample)

sample_proba = base_model.predict(sample_scaled)
sample_pred = np.argmax(sample_proba, axis=1)[0]

print("Przewidziana etykieta:", sample_pred)
print("Przewidziana nazwa gatunku:", class_names[sample_pred])

sample_proba_df = pd.DataFrame(sample_proba, columns=[f"P({name})" for name in class_names])
sample_proba_df

## 11. Odpowiedzi na pytania kontrolne

**1. Co oznacza warstwa Dense?**  
Warstwa `Dense` to warstwa w pełni połączona. Każdy neuron tej warstwy otrzymuje sygnał z każdego wejścia poprzedniej warstwy.

**2. Dlaczego ostatnia warstwa ma 3 neurony?**  
Ponieważ klasyfikujemy 3 gatunki irysów: `setosa`, `versicolor`, `virginica`.

**3. Do czego służy funkcja softmax?**  
`softmax` zamienia wyniki neuronów wyjściowych na rozkład prawdopodobieństwa po klasach. Suma prawdopodobieństw wynosi 1.

**4. Dlaczego używamy sparse_categorical_crossentropy?**  
Bo etykiety klas są zapisane jako liczby całkowite: 0, 1, 2. Nie są zakodowane w formacie one-hot.

**5. Co robi optimizer Adam?**  
Adam aktualizuje wagi sieci neuronowej podczas treningu. Łączy idee momentum i adaptacyjnego kroku uczenia.

**6. Co oznacza accuracy?**  
Accuracy to odsetek poprawnych predykcji wśród wszystkich predykcji.

**7. Co oznacza loss?**  
Loss to wartość funkcji straty, czyli miara błędu modelu. Im niższa wartość, tym lepiej model dopasowuje się do danych.

**8. Co robi funkcja argmax?**  
`argmax` wybiera indeks największej wartości. W klasyfikacji oznacza wybór klasy o najwyższym prawdopodobieństwie.

**9. Co pokazuje macierz pomyłek?**  
Macierz pomyłek pokazuje, ile przykładów każdej prawdziwej klasy zostało przypisanych do każdej klasy przewidzianej przez model.

**10. Czy model rzeczywiście rozumie kwiaty, czy tylko rozpoznaje zależności w danych?**  
Model nie rozumie kwiatów tak jak człowiek. Rozpoznaje wzorce liczbowe w danych i uczy się mapowania między cechami a etykietami klas.

## 12. Wnioski końcowe

1. Zbiór Iris jest mały, prosty i dobrze nadaje się do pierwszego eksperymentu z siecią neuronową.
2. Standaryzacja cech pomaga modelowi uczyć się stabilniej.
3. `sparse_categorical_crossentropy` jest właściwym wyborem, ponieważ klasy są zapisane jako liczby całkowite.
4. Model neuronowy zwykle osiąga wynik dużo lepszy niż losowe zgadywanie.
5. Najłatwiejszą klasą jest zwykle `setosa`, a najczęstsze pomyłki występują między `versicolor` i `virginica`.
6. Większy model nie zawsze jest lepszy. Na małym zbiorze danych prostsza architektura może być równie skuteczna.
7. Pełny proces uczenia modelu obejmuje nie tylko trening, ale też predykcję, analizę błędów, wykresy uczenia i interpretację wyników.